# CineInfini — User walkthrough

A typical user journey from install to leaderboard-ready output, run
inside a notebook. Mirrors the steps in
[`docs/QUICKSTART.md`](../docs/QUICKSTART.md).

This notebook **does not download any external weights** (it uses the
3 default-on modules which work without ArcFace/CLIP/DINOv2). For
the full audit experience, run `cineinfini bootstrap` in a terminal
first.


## 1. Setup

In [1]:
import os, sys, tempfile, json
from pathlib import Path

repo = Path(os.environ.get('CINEINFINI_REPO', '.'))
sys.path.insert(0, str(repo / 'src'))

import cineinfini
print(f"CineInfini {cineinfini.__version__}")


CineInfini 0.4.8.4


## 2. Generate three videos: good / mediocre / broken

Real audit usage = comparing many candidates. Let's make three
synthetic videos with different quality characteristics.


In [2]:
import numpy as np
import cv2

work = Path(tempfile.mkdtemp(prefix='cineinfini_walkthrough_'))
videos_dir = work / 'videos'
videos_dir.mkdir()

def make_video(path, kind):
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    w = cv2.VideoWriter(str(path), fourcc, 24.0, (320, 180))
    for i in range(48):
        f = np.full((180, 320, 3), 30, dtype=np.uint8)
        if kind == 'good':
            x = 20 + (i * 6) % 250
            f[60:120, x:x+40] = (220, 220, 50)
        elif kind == 'mediocre':
            # Slight flicker
            x = 20 + (i * 6) % 250
            f[60:120, x:x+40] = (200, 220, 50) if i%2==0 else (220, 200, 50)
        elif kind == 'broken':
            # Random noise = high flicker, low SSIM
            f = np.random.randint(0, 255, (180, 320, 3), dtype=np.uint8)
        w.write(f)
    w.release()

for kind in ('good', 'mediocre', 'broken'):
    p = videos_dir / f'{kind}.mp4'
    make_video(p, kind)
    print(f"  {p.name:15s} {p.stat().st_size:8d} B")


  good.mp4           10462 B
  mediocre.mp4       20213 B
  broken.mp4       1010701 B


## 3. Audit each one

In [3]:
import cineinfini.modules  # register
from cineinfini.core.config import default_config, set_config
from cineinfini.pipeline.orchestrator import run_audit
from cineinfini.aggregators import attach_videoscore_to_audit

cfg = default_config()
# Use ultralight profile-equivalent
cfg.processing['n_frames_per_shot'] = 4
cfg.paths['reports_dir'] = str(work / 'reports')
for m in cfg.modules:
    cfg.modules[m]['enabled'] = m in (
        'motion_coherence', 'identity_consistency', 'semantic_consistency',
    )
set_config(cfg)

results = {}
for video_path in sorted(videos_dir.iterdir()):
    audit_data, _ = run_audit(video_path, output_dir=work / 'reports' / video_path.stem)
    attach_videoscore_to_audit(audit_data)
    results[video_path.stem] = audit_data
    print(f"  {video_path.stem:10s} composite = "
          f"{audit_data['composite_score']:.3f}"
          if audit_data['composite_score'] is not None
          else "n/a")


  First pass: computing histogram differences (step=2)...


      First pass: 48/48 frames processed (elapsed 0.1s)
  [TIMING] First pass: 0.1s, 23 diffs
  Adaptive threshold: 0.164
  Second pass: detecting boundaries...
      Second pass: 48/48 frames processed (found 2 boundaries, elapsed 0.1s)
  [TIMING] Second pass: 0.1s, found 3 cut candidates
  [TIMING] Shot detection total: 0.2s, 1 shots generated
  Extracting 4 required frames...


  [TIMING] extract_shot_frames_global: 0.084s, 3 unique frames


semantic_consistency: CLIP unavailable (No module named 'clip')


  broken     composite = 0.107
  First pass: computing histogram differences (step=2)...
      First pass: 48/48 frames processed (elapsed 0.0s)
  [TIMING] First pass: 0.0s, 23 diffs
  Adaptive threshold: 0.080
  Second pass: detecting boundaries...
      Second pass: 48/48 frames processed (found 3 boundaries, elapsed 0.0s)
  [TIMING] Second pass: 0.0s, found 4 cut candidates
  [TIMING] Shot detection total: 0.0s, 1 shots generated
  Extracting 4 required frames...
      Extraction progress: 4/4 frames extracted
  [TIMING] extract_shot_frames_global: 0.073s, 4 unique frames


semantic_consistency: CLIP unavailable (No module named 'clip')


  good       composite = 0.963
  First pass: computing histogram differences (step=2)...
      First pass: 48/48 frames processed (elapsed 0.0s)
  [TIMING] First pass: 0.0s, 23 diffs
  Adaptive threshold: 0.082
  Second pass: detecting boundaries...
      Second pass: 48/48 frames processed (found 3 boundaries, elapsed 0.0s)
  [TIMING] Second pass: 0.0s, found 4 cut candidates
  [TIMING] Shot detection total: 0.0s, 1 shots generated
  Extracting 4 required frames...
      Extraction progress: 4/4 frames extracted
  [TIMING] extract_shot_frames_global: 0.072s, 4 unique frames


semantic_consistency: CLIP unavailable (No module named 'clip')


  mediocre   composite = 0.963


## 4. Compare side-by-side

In [4]:
print(f"{'video':12s} {'visual':>8s} {'temp':>8s} {'dyn':>8s} {'fact':>8s} {'composite':>10s}")
print('-' * 60)
for name, data in results.items():
    axes = data['videoscore_axes']
    composite = data['composite_score']
    def fmt(v): return f"{v:8.3f}" if v is not None else "     n/a"
    print(f"{name:12s} {fmt(axes['visual_quality'])} {fmt(axes['temporal_consistency'])} "
          f"{fmt(axes['dynamic_degree'])} {fmt(axes['factual_consistency'])} "
          f"{composite:10.3f}" if composite is not None else f"{name:12s} ...")


video          visual     temp      dyn     fact  composite
------------------------------------------------------------
broken            n/a    0.013    0.200      n/a      0.107
good              n/a    0.926    1.000      n/a      0.963
mediocre          n/a    0.927    1.000      n/a      0.963


The expected pattern:
- `good.mp4` → highest composite
- `mediocre.mp4` → middle (flicker drags temporal_consistency down)
- `broken.mp4` → lowest (every gate fails)

If your output matches this ordering, the pipeline is working
correctly end-to-end.


## 5. Export the best one to VBench format

In [5]:
from cineinfini.io.exporters import export_vbench_json

best = max(results.items(), key=lambda x: x[1].get('composite_score') or 0)
print(f"Best video: {best[0]}")

vbench_path = work / f"{best[0]}.vbench.json"
export_vbench_json(best[1], vbench_path)
payload = json.loads(vbench_path.read_text())
print(f"Measured: {len(payload['measured_dimensions'])}/16 VBench dimensions")
print(json.dumps(payload['scores'], indent=2)[:600])


Best video: mediocre
Measured: 2/16 VBench dimensions
{
  "subject_consistency": null,
  "background_consistency": 0.9268834871030295,
  "temporal_flickering": null,
  "motion_smoothness": null,
  "dynamic_degree": 1.0,
  "aesthetic_quality": null,
  "imaging_quality": null,
  "object_class": null,
  "multiple_objects": null,
  "human_action": null,
  "color": null,
  "spatial_relationship": null,
  "scene": null,
  "appearance_style": null,
  "temporal_style": null,
  "overall_consistency": null
}


## 6. Cleanup

In [6]:
import shutil
shutil.rmtree(work, ignore_errors=True)
print(f"Removed {work}")


Removed /tmp/cineinfini_walkthrough_5h0tg2sg


## Where to go from here

1. **Run on your real videos**: replace the synthetic generator above
   with a real video path.
2. **Use a fuller profile**: `cfg/profiles/postproduction.yaml` (9
   modules) or `academic.yaml` (21 modules) — needs `cineinfini
   bootstrap` first.
3. **Add competitor scores**: install `dover-vqa` + `fast-vqa`,
   `cineinfini bootstrap --include-optional`, enable the wrappers in
   the YAML config.
4. **Submit to a leaderboard**: feed the audit through
   `cineinfini export-vbench` and upload to
   [Vchitect/VBench](https://github.com/Vchitect/VBench).

For the full reference: [`docs/USER_MANUAL.md`](../docs/USER_MANUAL.md).
